# Steam Game Market Analysis - Database Edition

This notebook performs comprehensive analysis of Steam game market data using SQLite database operations.

**Features:**
- Memory-efficient database queries for large-scale data (50M+ reviews)
- Interactive visualizations and market insights
- Genre-based filtering and analysis
- User behavior and game popularity metrics
- Market concentration analysis

**Prerequisites:**
- Run the database assembler first: `cd graph-analyser/assemble-graph-database && make run`
- Ensure `steam_reviews.db` exists in the working directory or specify the path below

## Import Required Libraries

Import necessary libraries for database operations, data analysis, and visualization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import json
import warnings
from pathlib import Path
from typing import List, Dict, Tuple, Optional

# Configure display settings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print("✓ Libraries imported successfully")
print(f"✓ Pandas version: {pd.__version__}")
print(f"✓ Numpy version: {np.__version__}")

## Database Connection Setup

Connect to the Steam reviews SQLite database and verify its structure.

In [ ]:
# Database connection configuration
DB_PATH = "graph-analyser/assemble-graph-database/steam_reviews.db"

# Alternative paths to check
alternative_paths = [
    "steam_reviews.db",
    "graph-analyser/assemble-graph-database/steam_reviews.db",
    "./graph-analyser/assemble-graph-database/steam_reviews.db"
]

# Find the database file
db_path = None
for path in alternative_paths:
    if Path(path).exists():
        db_path = path
        break

if db_path is None:
    raise FileNotFoundError(
        "Steam reviews database not found. Please run the database assembler first:\n"
        "cd graph-analyser/assemble-graph-database && make run"
    )

print(f"✓ Database found at: {db_path}")

# Connect to database
try:
    conn = sqlite3.connect(db_path)
    print(f"✓ Connected to database: {db_path}")
    
    # Check database size
    db_size = Path(db_path).stat().st_size / (1024**2)  # MB
    print(f"✓ Database size: {db_size:.1f} MB")
    
except Exception as e:
    print(f"❌ Database connection failed: {e}")
    raise

## Database Schema Overview

Explore the database structure and verify data integrity.

In [ ]:
# Check database tables and their structure
tables_query = "SELECT name FROM sqlite_master WHERE type='table'"
tables = pd.read_sql_query(tables_query, conn)
print("📋 Database Tables:")
print(tables)

# Check table schemas
for table in tables['name'].values:
    print(f"\n📊 Table: {table}")
    schema_query = f"PRAGMA table_info({table})"
    schema = pd.read_sql_query(schema_query, conn)
    print(schema.to_string(index=False))

# Get basic statistics for each table
print("\n📈 Table Row Counts:")
stats = {}
for table in tables['name'].values:
    count_query = f"SELECT COUNT(*) as count FROM {table}"
    count = pd.read_sql_query(count_query, conn).iloc[0, 0]
    stats[table] = count
    print(f"   {table}: {count:,} rows")

# Display statistics as DataFrame for better formatting
stats_df = pd.DataFrame(list(stats.items()), columns=['Table', 'Row Count'])
print("\n📊 Summary:")
print(stats_df.to_string(index=False))

## Dataset Statistics and Overview

Calculate comprehensive statistics about the Steam market data.

In [ ]:
# Get comprehensive dataset statistics
def get_dataset_statistics(conn):
    """Get detailed statistics about the Steam dataset"""
    
    stats_queries = {
        'Total Games': "SELECT COUNT(*) FROM games",
        'Total Users': "SELECT COUNT(*) FROM users",
        'Total Reviews': "SELECT COUNT(*) FROM reviews",
        'Positive Reviews': "SELECT COUNT(*) FROM reviews WHERE voted_up = 1",
        'Negative Reviews': "SELECT COUNT(*) FROM reviews WHERE voted_up = 0", 
        'Free Game Reviews': "SELECT COUNT(*) FROM reviews WHERE received_for_free = 1",
        'Paid Game Reviews': "SELECT COUNT(*) FROM reviews WHERE received_for_free = 0",
        'Games with Reviews': """
            SELECT COUNT(DISTINCT game_id) FROM reviews
        """,
        'Active Users': """
            SELECT COUNT(DISTINCT user_id) FROM reviews
        """,
        'Avg Reviews per Game': """
            SELECT ROUND(AVG(review_count), 2) FROM (
                SELECT COUNT(*) as review_count FROM reviews GROUP BY game_id
            )
        """,
        'Avg Reviews per User': """
            SELECT ROUND(AVG(review_count), 2) FROM (
                SELECT COUNT(*) as review_count FROM reviews GROUP BY user_id  
            )
        """
    }
    
    results = {}
    for label, query in stats_queries.items():
        try:
            result = pd.read_sql_query(query, conn).iloc[0, 0]
            results[label] = result
        except Exception as e:
            print(f"Error with query '{label}': {e}")
            results[label] = "Error"
    
    return results

# Calculate and display statistics
print("🎮 Steam Game Market Statistics")
print("=" * 50)

stats = get_dataset_statistics(conn)
for key, value in stats.items():
    if isinstance(value, (int, float)):
        print(f"{key:.<30} {value:>15,}")
    else:
        print(f"{key:.<30} {value:>15}")

# Calculate additional metrics
positive_rate = (stats['Positive Reviews'] / stats['Total Reviews']) * 100 if stats['Total Reviews'] > 0 else 0
free_game_rate = (stats['Free Game Reviews'] / stats['Total Reviews']) * 100 if stats['Total Reviews'] > 0 else 0

print("\n📊 Key Metrics:")
print(f"{'Positive Review Rate':<30} {positive_rate:>14.1f}%")
print(f"{'Free Game Review Rate':<30} {free_game_rate:>14.1f}%")
print(f"{'Market Coverage':<30} {(stats['Games with Reviews']/stats['Total Games'])*100:>14.1f}%")

## Top Games Analysis

Analyze the most popular games by review count and rating.

In [ ]:
# Analyze top games by various metrics
def analyze_top_games(conn, limit=20, min_reviews=100):
    """Get top games by review count and positive rating"""
    
    query = f"""
    SELECT 
        g.name,
        g.game_id,
        COUNT(r.review_id) as review_count,
        AVG(CASE WHEN r.voted_up THEN 1.0 ELSE 0.0 END) as positive_rate,
        SUM(CASE WHEN r.voted_up THEN 1 ELSE 0 END) as positive_reviews,
        SUM(CASE WHEN NOT r.voted_up THEN 1 ELSE 0 END) as negative_reviews,
        SUM(CASE WHEN r.received_for_free THEN 1 ELSE 0 END) as free_reviews,
        g.genre_ids,
        g.categories_ids
    FROM games g 
    JOIN reviews r ON g.game_id = r.game_id 
    GROUP BY g.game_id, g.name, g.genre_ids, g.categories_ids
    HAVING review_count >= {min_reviews}
    ORDER BY review_count DESC 
    LIMIT {limit}
    """
    
    return pd.read_sql_query(query, conn)

# Get top games data
print("🏆 Top Games by Review Count")
print("=" * 60)

top_games = analyze_top_games(conn, limit=15)

# Display formatted results
for idx, row in top_games.iterrows():
    print(f"\n{idx+1:2d}. {row['name']}")
    print(f"    Reviews: {row['review_count']:,}")
    print(f"    Positive Rate: {row['positive_rate']:.1%}")
    print(f"    Free Reviews: {row['free_reviews']:,} ({row['free_reviews']/row['review_count']:.1%})")

# Show summary statistics
print(f"\n📊 Summary (Top {len(top_games)} games):")
print(f"Total Reviews: {top_games['review_count'].sum():,}")
print(f"Average Reviews per Game: {top_games['review_count'].mean():.0f}")
print(f"Average Positive Rate: {top_games['positive_rate'].mean():.1%}")

# Store for visualization
top_games_data = top_games.copy()

## Data Visualization

Create comprehensive visualizations of the Steam market data.

In [ ]:
# Create comprehensive visualizations
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Steam Game Market Analysis Dashboard', fontsize=16, y=0.98)

# 1. Dataset Overview (Top Left)
labels = ['Games', 'Users', 'Reviews'] 
counts = [stats['Total Games'], stats['Total Users'], stats['Total Reviews']]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

bars = ax1.bar(labels, counts, color=colors)
ax1.set_title('Dataset Size Overview')
ax1.set_ylabel('Count')
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1e6:.1f}M' if x >= 1e6 else f'{x/1e3:.0f}K'))

# Add value labels on bars
for bar, count in zip(bars, counts):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
             f'{count:,.0f}', ha='center', va='bottom')

# 2. Review Sentiment Distribution (Top Right)
sentiment_data = [stats['Positive Reviews'], stats['Negative Reviews']]
sentiment_labels = ['Positive', 'Negative']
sentiment_colors = ['#2ca02c', '#d62728']

wedges, texts, autotexts = ax2.pie(sentiment_data, labels=sentiment_labels, 
                                  colors=sentiment_colors, autopct='%1.1f%%', 
                                  startangle=90)
ax2.set_title('Review Sentiment Distribution')

# 3. Top Games by Review Count (Bottom Left)
top_10_games = top_games_data.head(10)
y_pos = np.arange(len(top_10_games))

bars = ax3.barh(y_pos, top_10_games['review_count'], color='#9467bd')
ax3.set_yticks(y_pos)
ax3.set_yticklabels([name[:25] + '...' if len(name) > 25 else name for name in top_10_games['name']])
ax3.set_xlabel('Review Count')
ax3.set_title('Top 10 Games by Review Count')
ax3.invert_yaxis()

# Add value labels
for i, (idx, row) in enumerate(top_10_games.iterrows()):
    ax3.text(row['review_count'], i, f" {row['review_count']:,}", 
            va='center', fontsize=9)

# 4. Review Count vs Positive Rate Scatter (Bottom Right)
scatter = ax4.scatter(top_games_data['review_count'], 
                     top_games_data['positive_rate'], 
                     alpha=0.6, s=60, c='#ff7f0e')
ax4.set_xlabel('Review Count')
ax4.set_ylabel('Positive Rate')
ax4.set_title('Review Count vs Positive Rate')
ax4.set_xscale('log')
ax4.grid(True, alpha=0.3)

# Format axes
ax4.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x:.1%}'))

plt.tight_layout()
plt.show()

# Create output directory if it doesn't exist
output_dir = Path("analysis_plots")
output_dir.mkdir(exist_ok=True)

# Save the plot
plt.savefig(output_dir / 'steam_market_dashboard.png', dpi=300, bbox_inches='tight')
print(f"📊 Dashboard saved to {output_dir / 'steam_market_dashboard.png'}")

## Genre-Based Analysis

Analyze games by genre categories and market segments.

In [ ]:
# Genre analysis function
def analyze_genre(conn, genre_id, min_reviews=50):
    """Analyze games by specific genre"""
    
    query = f"""
    SELECT 
        g.name,
        g.game_id,
        COUNT(r.review_id) as review_count,
        AVG(CASE WHEN r.voted_up THEN 1.0 ELSE 0.0 END) as positive_rate,
        g.genre_ids
    FROM games g
    JOIN reviews r ON g.game_id = r.game_id
    WHERE g.genre_ids LIKE '%"{genre_id}"%'
    GROUP BY g.game_id, g.name, g.genre_ids
    HAVING review_count >= {min_reviews}
    ORDER BY positive_rate DESC
    LIMIT 20
    """
    
    return pd.read_sql_query(query, conn)

# Common Steam genre IDs (examples - you may need to adjust based on your data)
genre_mapping = {
    "1": "Action",
    "2": "Strategy", 
    "3": "RPG",
    "4": "Simulation",
    "9": "Racing",
    "10": "Sports",
    "23": "Indie"
}

# Analyze each genre
print("🎯 Genre-Based Market Analysis")
print("=" * 60)

genre_results = {}
for genre_id, genre_name in genre_mapping.items():
    try:
        genre_games = analyze_genre(conn, genre_id, min_reviews=20)
        
        if len(genre_games) > 0:
            genre_results[genre_name] = {
                'games_count': len(genre_games),
                'total_reviews': genre_games['review_count'].sum(),
                'avg_positive_rate': genre_games['positive_rate'].mean(),
                'top_game': genre_games.iloc[0]['name'] if len(genre_games) > 0 else 'N/A'
            }
            
            print(f"\n📊 {genre_name} Games (Genre ID: {genre_id})")
            print(f"   Games analyzed: {len(genre_games)}")
            print(f"   Total reviews: {genre_games['review_count'].sum():,}")
            print(f"   Avg positive rate: {genre_games['positive_rate'].mean():.1%}")
            print(f"   Top rated game: {genre_games.iloc[0]['name']}")
        else:
            print(f"\n❌ No games found for {genre_name} (Genre ID: {genre_id})")
            
    except Exception as e:
        print(f"\n❌ Error analyzing {genre_name}: {e}")

# Create genre comparison visualization if we have results
if genre_results:
    print("\n📈 Creating genre comparison visualization...")
    
    genre_df = pd.DataFrame(genre_results).T
    genre_df.reset_index(inplace=True)
    genre_df.rename(columns={'index': 'genre'}, inplace=True)
    
    # Genre comparison plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Games count by genre
    ax1.bar(genre_df['genre'], genre_df['games_count'], color='skyblue')
    ax1.set_title('Number of Popular Games by Genre')
    ax1.set_ylabel('Number of Games')
    ax1.tick_params(axis='x', rotation=45)
    
    # Average positive rate by genre
    ax2.bar(genre_df['genre'], genre_df['avg_positive_rate'], color='lightcoral')
    ax2.set_title('Average Positive Review Rate by Genre')
    ax2.set_ylabel('Positive Rate')
    ax2.tick_params(axis='x', rotation=45)
    ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x:.1%}'))
    
    plt.tight_layout()
    plt.show()
    
    # Save genre analysis
    plt.savefig(output_dir / 'genre_analysis.png', dpi=300, bbox_inches='tight')
    print(f"📊 Genre analysis saved to {output_dir / 'genre_analysis.png'}")
    
    # Export genre data to CSV
    genre_df.to_csv(output_dir / 'genre_analysis.csv', index=False)
    print(f"💾 Genre data exported to {output_dir / 'genre_analysis.csv'}")
else:
    print("\n⚠️  No genre data available for visualization")

## User Behavior Analysis

Analyze user review patterns and activity levels.

In [ ]:
# User behavior analysis
print("👥 User Behavior Analysis")
print("=" * 50)

# User activity distribution
user_activity_query = """
SELECT 
    games_reviewed,
    COUNT(*) as user_count
FROM (
    SELECT user_id, COUNT(DISTINCT game_id) as games_reviewed
    FROM reviews 
    GROUP BY user_id
    HAVING games_reviewed <= 50  -- Focus on users with reasonable activity
) user_stats
GROUP BY games_reviewed
ORDER BY games_reviewed
"""

user_activity = pd.read_sql_query(user_activity_query, conn)
print(f"📊 User Activity Distribution (users who reviewed ≤50 games):")
print(user_activity.head(15).to_string(index=False))

# Most active users
most_active_query = """
SELECT 
    user_id,
    COUNT(DISTINCT game_id) as games_reviewed,
    COUNT(*) as total_reviews,
    AVG(CASE WHEN voted_up THEN 1.0 ELSE 0.0 END) as positivity_rate,
    SUM(CASE WHEN received_for_free THEN 1 ELSE 0 END) as free_game_reviews
FROM reviews 
GROUP BY user_id 
ORDER BY games_reviewed DESC 
LIMIT 10
"""

most_active = pd.read_sql_query(most_active_query, conn)
print(f"\n🏆 Top 10 Most Active Users:")
print(most_active.to_string(index=False))

# User sentiment patterns
user_sentiment_query = """
SELECT 
    CASE 
        WHEN positivity_rate >= 0.8 THEN 'Very Positive (80%+)'
        WHEN positivity_rate >= 0.6 THEN 'Positive (60-79%)'
        WHEN positivity_rate >= 0.4 THEN 'Mixed (40-59%)'
        ELSE 'Negative (<40%)'
    END as sentiment_category,
    COUNT(*) as user_count,
    AVG(review_count) as avg_reviews
FROM (
    SELECT 
        user_id,
        COUNT(*) as review_count,
        AVG(CASE WHEN voted_up THEN 1.0 ELSE 0.0 END) as positivity_rate
    FROM reviews 
    GROUP BY user_id
    HAVING review_count >= 5  -- Users with at least 5 reviews
) user_sentiment
GROUP BY sentiment_category
ORDER BY 
    CASE sentiment_category
        WHEN 'Very Positive (80%+)' THEN 1
        WHEN 'Positive (60-79%)' THEN 2
        WHEN 'Mixed (40-59%)' THEN 3
        ELSE 4
    END
"""

user_sentiment = pd.read_sql_query(user_sentiment_query, conn)
print(f"\n😊 User Sentiment Distribution (users with 5+ reviews):")
print(user_sentiment.to_string(index=False))

# Visualize user behavior
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# User activity distribution (log scale)
activity_subset = user_activity[user_activity['games_reviewed'] <= 20]  # Focus on reasonable range
ax1.bar(activity_subset['games_reviewed'], activity_subset['user_count'], alpha=0.7, color='steelblue')
ax1.set_xlabel('Number of Games Reviewed')
ax1.set_ylabel('Number of Users')
ax1.set_title('User Activity Distribution')
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3)

# User sentiment distribution
ax2.pie(user_sentiment['user_count'], labels=user_sentiment['sentiment_category'], 
        autopct='%1.1f%%', startangle=90)
ax2.set_title('User Sentiment Distribution\n(Users with 5+ reviews)')

plt.tight_layout()
plt.show()

# Save user behavior analysis
plt.savefig(output_dir / 'user_behavior_analysis.png', dpi=300, bbox_inches='tight')
print(f"📊 User behavior analysis saved to {output_dir / 'user_behavior_analysis.png'}")

# Export user data
user_activity.to_csv(output_dir / 'user_activity_distribution.csv', index=False)
user_sentiment.to_csv(output_dir / 'user_sentiment_distribution.csv', index=False)
print(f"💾 User data exported to CSV files")

## Market Concentration Analysis

Analyze market concentration using Gini coefficient and Lorenz curves.

In [ ]:
# Market concentration analysis
def calculate_gini_coefficient(data):
    """Calculate Gini coefficient for measuring inequality"""
    data = np.array(data)
    data = np.sort(data)
    n = len(data)
    cumsum = np.cumsum(data)
    
    # Gini coefficient formula
    gini = (n + 1 - 2 * np.sum(cumsum) / cumsum[-1]) / n
    return gini

def create_lorenz_curve(data, title="Lorenz Curve"):
    """Create Lorenz curve for inequality visualization"""
    data = np.sort(data)
    n = len(data)
    
    # Calculate cumulative percentages
    cumulative_population = np.arange(1, n + 1) / n
    cumulative_wealth = np.cumsum(data) / np.sum(data)
    
    return cumulative_population, cumulative_wealth

print("📊 Market Concentration Analysis")
print("=" * 50)

# Get review distribution by game
game_reviews_query = """
SELECT 
    game_id,
    COUNT(*) as review_count
FROM reviews 
GROUP BY game_id 
ORDER BY review_count DESC
"""

game_reviews = pd.read_sql_query(game_reviews_query, conn)
review_counts = game_reviews['review_count'].values

# Calculate market concentration metrics
gini_coeff = calculate_gini_coefficient(review_counts)
total_reviews = review_counts.sum()

# Top percentiles
top_1_percent = int(len(review_counts) * 0.01)
top_5_percent = int(len(review_counts) * 0.05)
top_10_percent = int(len(review_counts) * 0.10)

top_1_share = review_counts[:top_1_percent].sum() / total_reviews
top_5_share = review_counts[:top_5_percent].sum() / total_reviews 
top_10_share = review_counts[:top_10_percent].sum() / total_reviews

print(f"📈 Market Concentration Metrics:")
print(f"{'Gini Coefficient':<25} {gini_coeff:.3f}")
print(f"{'Top 1% games share':<25} {top_1_share:.1%}")
print(f"{'Top 5% games share':<25} {top_5_share:.1%}")
print(f"{'Top 10% games share':<25} {top_10_share:.1%}")

print(f"\n🎮 Review Distribution:")
print(f"{'Total games with reviews':<25} {len(review_counts):,}")
print(f"{'Total reviews':<25} {total_reviews:,}")
print(f"{'Average reviews per game':<25} {total_reviews/len(review_counts):.1f}")
print(f"{'Median reviews per game':<25} {np.median(review_counts):.0f}")

# Create Lorenz curve visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Lorenz curve
population_pct, wealth_pct = create_lorenz_curve(review_counts)
ax1.plot(population_pct, wealth_pct, 'b-', linewidth=2, label='Lorenz Curve')
ax1.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect Equality')
ax1.fill_between(population_pct, wealth_pct, alpha=0.3)
ax1.set_xlabel('Cumulative Share of Games')
ax1.set_ylabel('Cumulative Share of Reviews') 
ax1.set_title(f'Lorenz Curve - Review Distribution\nGini Coefficient: {gini_coeff:.3f}')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Review distribution histogram (log scale)
ax2.hist(review_counts, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
ax2.set_xlabel('Reviews per Game')
ax2.set_ylabel('Number of Games')
ax2.set_title('Distribution of Reviews per Game')
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Save concentration analysis
plt.savefig(output_dir / 'market_concentration_analysis.png', dpi=300, bbox_inches='tight')
print(f"📊 Market concentration analysis saved to {output_dir / 'market_concentration_analysis.png'}")

# Export concentration metrics
concentration_metrics = {
    'Metric': ['Gini Coefficient', 'Top 1% Share', 'Top 5% Share', 'Top 10% Share', 
               'Total Games', 'Total Reviews', 'Avg Reviews/Game', 'Median Reviews/Game'],
    'Value': [gini_coeff, top_1_share, top_5_share, top_10_share,
              len(review_counts), total_reviews, total_reviews/len(review_counts), np.median(review_counts)]
}

concentration_df = pd.DataFrame(concentration_metrics)
concentration_df.to_csv(output_dir / 'market_concentration_metrics.csv', index=False)
print(f"💾 Concentration metrics exported to {output_dir / 'market_concentration_metrics.csv'}")

## Custom Queries and Analysis

Execute custom SQL queries for specific research questions.

In [ ]:
# Custom analysis queries
print("🔍 Custom Analysis Queries")
print("=" * 50)

# 1. Games with most polarized reviews (high volume, mixed sentiment)
polarized_games_query = """
SELECT 
    g.name,
    g.game_id,
    COUNT(r.review_id) as total_reviews,
    AVG(CASE WHEN r.voted_up THEN 1.0 ELSE 0.0 END) as positive_rate,
    ABS(AVG(CASE WHEN r.voted_up THEN 1.0 ELSE 0.0 END) - 0.5) as polarization_score
FROM games g 
JOIN reviews r ON g.game_id = r.game_id 
GROUP BY g.game_id, g.name 
HAVING total_reviews >= 1000
ORDER BY polarization_score ASC, total_reviews DESC
LIMIT 10
"""

polarized_games = pd.read_sql_query(polarized_games_query, conn)
print("🎭 Most Polarized Games (mixed sentiment with high volume):")
print(polarized_games.to_string(index=False))

# 2. Free vs Paid games performance comparison
free_vs_paid_query = """
SELECT 
    CASE WHEN r.received_for_free THEN 'Free' ELSE 'Paid' END as game_type,
    COUNT(DISTINCT g.game_id) as unique_games,
    COUNT(r.review_id) as total_reviews,
    AVG(CASE WHEN r.voted_up THEN 1.0 ELSE 0.0 END) as avg_positive_rate,
    AVG(review_count) as avg_reviews_per_game
FROM games g 
JOIN reviews r ON g.game_id = r.game_id 
JOIN (
    SELECT game_id, COUNT(*) as review_count 
    FROM reviews 
    GROUP BY game_id
) rc ON g.game_id = rc.game_id
GROUP BY r.received_for_free
"""

free_vs_paid = pd.read_sql_query(free_vs_paid_query, conn)
print(f"\n💰 Free vs Paid Games Comparison:")
print(free_vs_paid.to_string(index=False))

# 3. Network analysis - Games often reviewed together
shared_reviewers_query = """
SELECT 
    g1.name as game1_name,
    g2.name as game2_name,
    shared_users as shared_reviewers,
    g1_reviews,
    g2_reviews,
    ROUND(shared_users * 1.0 / MIN(g1_reviews, g2_reviews), 3) as overlap_ratio
FROM (
    SELECT 
        r1.game_id as game1_id,
        r2.game_id as game2_id,
        COUNT(*) as shared_users
    FROM reviews r1 
    JOIN reviews r2 ON r1.user_id = r2.user_id AND r1.game_id < r2.game_id
    GROUP BY r1.game_id, r2.game_id 
    HAVING shared_users >= 50
    ORDER BY shared_users DESC
    LIMIT 15
) shared
JOIN (SELECT game_id, COUNT(*) as g1_reviews FROM reviews GROUP BY game_id) c1 
    ON shared.game1_id = c1.game_id
JOIN (SELECT game_id, COUNT(*) as g2_reviews FROM reviews GROUP BY game_id) c2 
    ON shared.game2_id = c2.game_id
JOIN games g1 ON shared.game1_id = g1.game_id
JOIN games g2 ON shared.game2_id = g2.game_id
ORDER BY shared_users DESC
"""

try:
    shared_reviewers = pd.read_sql_query(shared_reviewers_query, conn)
    print(f"\n🤝 Games with Most Shared Reviewers:")
    print(shared_reviewers.to_string(index=False))
except Exception as e:
    print(f"\n⚠️  Network analysis query too complex for current dataset: {e}")

# 4. Temporal patterns (if we had date data - placeholder for future enhancement)
print(f"\n📅 Temporal Analysis:")
print("Note: Temporal analysis would require review timestamp data")
print("Future enhancement: Add review_date field to analyze trends over time")

# Custom query playground - Users can modify this
print(f"\n🛠️ Custom Query Playground:")
print("Modify the query below to explore specific research questions:")

# Example: Users can change this query
custom_query = """
SELECT 
    g.name,
    COUNT(r.review_id) as review_count,
    AVG(CASE WHEN r.voted_up THEN 1.0 ELSE 0.0 END) as positive_rate
FROM games g 
JOIN reviews r ON g.game_id = r.game_id 
WHERE g.name LIKE '%Counter%' OR g.name LIKE '%Strike%'
GROUP BY g.game_id, g.name 
ORDER BY review_count DESC
"""

try:
    custom_result = pd.read_sql_query(custom_query, conn)
    print("🎯 Custom Query Results (Counter-Strike related games):")
    print(custom_result.to_string(index=False))
except Exception as e:
    print(f"❌ Custom query error: {e}")

## Summary and Export Results

Summarize key findings and export analysis results.

In [ ]:
# Summary of key findings
print("🎯 Steam Game Market Analysis - Key Findings")
print("=" * 60)

# Compile key statistics
key_findings = {
    "Dataset Scale": {
        "Total Games": f"{stats['Total Games']:,}",
        "Total Users": f"{stats['Total Users']:,}",
        "Total Reviews": f"{stats['Total Reviews']:,}",
    },
    "Market Sentiment": {
        "Positive Review Rate": f"{positive_rate:.1f}%",
        "Free Game Review Rate": f"{free_game_rate:.1f}%",
    },
    "Market Concentration": {
        "Gini Coefficient": f"{gini_coeff:.3f}",
        "Top 1% Games Market Share": f"{top_1_share:.1%}",
        "Top 10% Games Market Share": f"{top_10_share:.1%}",
    },
    "Top Performing Game": {
        "Most Reviewed": top_games_data.iloc[0]['name'] if len(top_games_data) > 0 else "N/A",
        "Review Count": f"{top_games_data.iloc[0]['review_count']:,}" if len(top_games_data) > 0 else "N/A",
        "Positive Rate": f"{top_games_data.iloc[0]['positive_rate']:.1%}" if len(top_games_data) > 0 else "N/A"
    }
}

for category, metrics in key_findings.items():
    print(f"\n📊 {category}:")
    for metric, value in metrics.items():
        print(f"   {metric:<25} {value}")

# Export comprehensive results
print(f"\n💾 Exporting Analysis Results:")
print(f"   Output directory: {output_dir}")

# List all generated files
output_files = list(output_dir.glob("*"))
for file_path in sorted(output_files):
    file_size = file_path.stat().st_size / 1024  # KB
    print(f"   ✓ {file_path.name} ({file_size:.1f} KB)")

# Create a summary report
summary_report = {
    "Analysis Date": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
    "Database Path": db_path,
    "Database Size (MB)": f"{db_size:.1f}",
    **{f"Dataset.{k}": v for k, v in key_findings["Dataset Scale"].items()},
    **{f"Sentiment.{k}": v for k, v in key_findings["Market Sentiment"].items()},
    **{f"Concentration.{k}": v for k, v in key_findings["Market Concentration"].items()},
    **{f"TopGame.{k}": v for k, v in key_findings["Top Performing Game"].items()},
}

# Save summary as JSON and CSV
summary_df = pd.DataFrame([summary_report])
summary_df.to_csv(output_dir / 'analysis_summary.csv', index=False)

import json
with open(output_dir / 'analysis_summary.json', 'w') as f:
    json.dump(summary_report, f, indent=2)

print(f"\n✅ Analysis Complete!")
print(f"📁 All results saved to: {output_dir}")
print(f"📊 Generated {len(output_files)} files")
print(f"🎮 Key insight: Market shows {gini_coeff:.3f} Gini coefficient concentration")
print(f"⭐ Top game: {top_games_data.iloc[0]['name']} with {top_games_data.iloc[0]['review_count']:,} reviews")

## Close Database Connection

Properly close the database connection and clean up resources.

In [ ]:
# Close database connection and cleanup
try:
    if conn:
        conn.close()
        print("✅ Database connection closed successfully")
    else:
        print("ℹ️  No database connection to close")
        
except Exception as e:
    print(f"⚠️  Error closing database connection: {e}")

# Final memory cleanup
import gc
gc.collect()

print(f"\n🎉 Steam Market Analysis Complete!")
print(f"📊 Generated comprehensive analysis with visualizations")
print(f"💾 Results exported to: {output_dir}")
print(f"🔬 Ready for further research and insights")

# Instructions for next steps
print(f"\n📋 Next Steps:")
print(f"   1. Review generated plots in {output_dir}")
print(f"   2. Examine exported CSV data for detailed analysis") 
print(f"   3. Modify custom queries above for specific research questions")
print(f"   4. Use genre filtering for targeted market segments")
print(f"   5. Consider network analysis using shared reviewer data")